# B2.1 · Plan–act–verify

**Function B — Product & Application Security → The Security Automation / Harness Engineer**  ·  *AI for Security*

---

**Risk.** Writing the code instead of writing the loop.

**Control.** The minimum viable security harness: context, toolset, verifier, budget.

**This lab.** One scaffold, swappable model, unchanged loop.

| | |
|---|---|
| Open-source tooling | Python, LiteLLM |
| Open-weight models | GLM-4.6, Llama 3.3, Kimi K2 |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("B2.1"))

Plan–act–verify is the whole harness. Build it once, deliberately, and every later lesson in this track is a modification to one of the three.

In [ ]:
from cybercommons import loop

TARGET = "def add(a, b): return a + b"
attempts = ["def add(a, b): return a - b",      # wrong
            "def add(a, b): return a * b",      # still wrong
            TARGET]                             # right

trace = loop.run(loop.FakeModel(attempts), loop.oracle(TARGET),
                 goal="implement add", max_steps=5)
print(trace.table())

Three plans, three acts, three verifications, one stop. Now watch what changes when the verifier is the only thing you swap.

In [ ]:
weak = loop.run(loop.FakeModel(attempts), loop.llm_judge(),
                goal="implement add", max_steps=5)
print(weak.table())
print("\nSame model, same proposals. The harness stopped on attempt 1 with"
      "\nsubtraction and called it a success.")

### Expect

The oracle run takes three steps and succeeds on the correct implementation. The judge run stops on step 1 with `return a - b` and reports success.

### Your turn

Add a fourth move to the loop: *reflect* — feed the verifier's failure detail back into the next proposal. Does it help when the verifier is an oracle? Does it help when the verifier is a judge?

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/B2.1.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*